# Subject 1 · Sentiment analysis: VADER reads readers' reviews

**Course "Artificial Intelligence"** · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa
**Mini-project, subject 1** (sentiment analysis) · **Chapter 3** (symbolic AI: readable rules; Transformers) · **Duration:** about 1 h 30

**What you will do**
- gather a small corpus of 20 readers' reviews of novels, each labelled by hand: positive, negative or neutral;
- score each review with VADER, the lexicon-based tool named in the syllabus: scored words, negation ("not", "never"), intensifiers ("very", "really");
- measure VADER's accuracy against the human reading, then analyze its errors: negation, irony, words missing from the lexicon.

**How to use this notebook**
Run each cell from top to bottom: click the ▶ button at its left, or press Shift+Enter. You never have to write code. Change only the fields of the cells titled "TO CHANGE". In Colab, the menu "Runtime → Run all" runs the whole notebook at once.

Before each code cell, a question asks you to **predict** the result: write down your answer, run the cell, then compare with the section "What you should see".

> **Fictional data.** The 20 reviews in this notebook were invented for the exercise, and their "manual" labels are those of the author of the notebook. No real reader is quoted: never present these figures as survey results.

## Step 1 · Load the tools

This cell installs **VADER** (the `vaderSentiment` library), the sentiment analysis tool named in the syllabus. It is designed for English, and it is the tool of this notebook. The cell also loads pandas (tables), matplotlib (charts) and two small Python modules: `io` and `re`.

**Before you run it:** will this cell already analyze a text, or only prepare the tools?

In [ ]:
#@title Install VADER and load the tools
%pip install -q vaderSentiment
import io                                  # read a text as if it were a file
import re                                  # split a sentence into words
import pandas as pd                        # data tables
import matplotlib.pyplot as plt            # charts
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()    # VADER, the tool of this notebook
pd.set_option("display.max_colwidth", 60)  # show the reviews in full
print("Tools loaded.")

**What you should see:** sometimes a line about the installation, then `Tools loaded.` Nothing is analyzed yet: the cell prepares the work.

**Question:** VADER is designed for English. What do you think will happen if it reads a review written in French? Write down your answer: you will check it in step 17.

## Step 2 · The corpus (1/2): ten ordinary reviews

Here are ten readers' reviews of novels, **invented for the exercise**. Each line holds three pieces of information separated by a semicolon, as in a CSV file:
- the **text** of the review;
- the **manual** label: the human reading (positive, negative or neutral). It is the reference we will compare the machine with;
- the **difficulty** expected for the machine: `plain` or `negation`.

pandas reads this text as if it were a file and turns it into a table.

**Before you run it:** read the ten reviews in the cell. Do you agree with each label? Which one looks the hardest for a machine?

In [ ]:
#@title The corpus (1/2): ten "ordinary" reviews (fictional data)
# One line per review: text; manual label; expected difficulty
ordinary_reviews = """A magnificent book, I adored it.;positive;plain
A gripping story, very well written.;positive;plain
Boring book, and far too long.;negative;plain
Heavy style, weak plot: what a disappointment.;negative;plain
The novel has 300 pages and is set in Paris.;neutral;plain
Read for my comparative literature class.;neutral;plain
I did not like this book.;negative;negation
Not bad at all, I recommend it.;positive;negation
I never found it interesting.;negative;negation
I cannot say that this book is good.;negative;negation"""
columns = ["text", "manual", "difficulty"]
ordinary = pd.read_csv(io.StringIO(ordinary_reviews), sep=";", names=columns)
ordinary

**What you should see:** a table of 10 rows numbered from 0 to 9 (Python counts from 0) and 3 columns: `text`, `manual`, `difficulty`. It holds 3 positive reviews, 5 negative ones and 2 neutral ones.

**Question:** is "Read for my comparative literature class." really neutral? What decides your label: the words of the review, or what you imagine about its author?

## Step 3 · The corpus (2/2): ten tricky reviews

Ten more reviews, also invented, contain well-known traps for sentiment analysis:
- **irony**: positive words used to mean the opposite;
- **missing words**, absent from VADER's lexicon ("page-turner", "breathless") — plus one emoticon, to see whether VADER knows it;
- a **sad theme** (war, death) that is not necessarily a negative judgment on the book.

The cell puts the 20 reviews together in one table, `df`, then counts the manual labels.

**Before you run it:** out of the 20 reviews, how many are positive, negative and neutral?

In [ ]:
#@title The corpus (2/2): ten "tricky" reviews (fictional data)
# Traps: irony, word missing from the lexicon, sad theme
tricky_reviews = """Great, another 900 pages where nothing happens.;negative;irony
Well done, author: I fell asleep on page 10.;negative;irony
What a joy to reread the same scene ten times.;negative;irony
A page-turner: I devoured it in one night!;positive;missing word
Predictable plot, cardboard characters.;negative;missing word
Chiselled prose, breathless chapters.;positive;missing word
Finished it last night :);positive;missing word
War, fear and death: a masterpiece.;positive;sad theme
The book tells of the father's death in the war.;neutral;sad theme
A sad story, but so beautiful.;positive;sad theme"""
tricky = pd.read_csv(io.StringIO(tricky_reviews), sep=";", names=columns)
df = pd.concat([ordinary, tricky], ignore_index=True)   # the 20 reviews, from 0 to 19
df["manual"].value_counts()

**What you should see:** the count of the manual labels: `negative 9`, `positive 8`, `neutral 3` (20 reviews in all).

**Question:** neutral reviews are rare here (3 out of 20). Is it also the case in the reviews you read online? Why does a reader rarely write a neutral review?

## Step 4 · The lexicon: words and their scores

VADER (*Valence Aware Dictionary and sEntiment Reasoner*, Hutto & Gilbert, 2014) does not understand sentences: it looks up each word in a list of **scored words**, from −4 (very negative) to +4 (very positive). Each score is the average of the ratings given by ten human raters. VADER was built for social media texts, so its lexicon also contains emoticons and slang. Two more lists come with it:
- the **negations** (`not`, `never`, `without`, `isn't`…), which reverse the word that follows;
- the **intensifiers**, or *boosters* (`very`, `really`, `extremely`…, and also `slightly`, which weakens).

This is **symbolic AI** (chapter 3): knowledge written by hand, readable by a human. The demonstration on the course site ("Read a sentence like a machine") works the same way, with a much smaller lexicon.

**Before you run it:** how many words does VADER's lexicon contain: about 100, about 1,000, or about 7,500?

In [ ]:
#@title VADER's lexicon: words and their scores (from -4 to +4)
from vaderSentiment.vaderSentiment import NEGATE, BOOSTER_DICT
LEXICON = analyzer.lexicon      # VADER's word list: {word: score}
NEGATIONS = NEGATE              # "not", "never", "without"...
INTENSIFIERS = BOOSTER_DICT     # "very", "really", "slightly"...
print("Words in the lexicon:", len(LEXICON))
print("Negations:", len(NEGATIONS), "| intensifiers:", len(INTENSIFIERS))

**What you should see:** `Words in the lexicon: 7506`, then `Negations: 59 | intensifiers: 84`. About 7,500 entries: it is large, but it was rated for tweets and online comments, not for literary criticism.

**Question:** which words of literary criticism would you expect to be missing from a lexicon built for social media? Give two examples, then check them in the next step.

## Step 5 · Look up a word in the lexicon

This cell shows the score of a few words, then the score of the word you type in the field `word` (on the right of the cell in Colab). "absent from the lexicon" means that VADER will ignore the word: it will count as zero.

**Before you run it:** does the word "novel" have a score? And the emoticon ":)"? And "sublime"?

In [ ]:
#@title TO CHANGE: look up a word in the lexicon
word = "sublime" #@param {type:"string"}
for w in ["good", "boring", "novel", ":)", word]:
    print(w, "->", LEXICON.get(w.lower(), "absent from the lexicon"))

**What you should see:** `good -> 1.9`, `boring -> -1.3`, `novel -> 1.3`, `:) -> 2.0`, `sublime -> absent from the lexicon`.

Two surprises. "Sublime", a key word of literary criticism, is missing. And "novel" is scored +1.3: in VADER's lexicon it is the **adjective** *novel* (new, original: "a novel idea"), not the noun (a book). The tool cannot tell which meaning is used.

**Question:** type "page-turner", then "masterpiece", then "meh" in the field. A missing word counts as zero: what score will a review get if VADER knows none of its words?

## Step 6 · Split a sentence into words

To look up the words, we first split the sentence: everything goes to lower case, the punctuation disappears, but the emoticons `:)` and `:(` are kept. The function `known_words` returns the words of the sentence that are in VADER's lexicon, with their score. (VADER splits sentences in a very similar way.)

**Before you run it:** into how many pieces will the sentence "I didn't like this book :(" be split? Which ones are in the lexicon?

In [ ]:
#@title Split a sentence into words, find the words known to the lexicon
def split_words(sentence):
    sentence = sentence.lower().replace("’", "'")   # lower case, straight apostrophe
    # keep the words (letters, digits, - and ') and the emoticons :) and :(
    return re.findall(r":\)|:\(|[\w'-]+", sentence)

def known_words(sentence):
    found = []
    for word in split_words(sentence):
        if word in LEXICON:
            found.append((word, LEXICON[word]))   # (word, score)
    return found

print(split_words("I didn't like this book :("))
print(known_words("I didn't like this book :("))

**What you should see:** `['i', "didn't", 'like', 'this', 'book', ':(']`, 6 pieces, then `[('like', 1.5), (':(', -1.9)]`: two known words, one of them the emoticon.

**Question:** "like" is recognized with a positive score (+1.5), although the review is negative. Which word of the sentence must reverse this score? Is it in the lexicon, or in another list?

## Step 7 · The method: the score of a sentence

VADER's method, simplified (Hutto & Gilbert, 2014):
1. it adds up the scores of the words known to the lexicon;
2. an intensifier before a word ("very good") makes it stronger, by about 0.3;
3. a negation among the **three words before** a word ("not good", "did not like") reverses its score and weakens it: it is multiplied by −0.74;
4. after "but", words count 1.5 times more, and before it only half; exclamation marks and CAPITAL LETTERS add strength;
5. the total is brought back between −1 and +1 by the formula total / √(total² + 15): this is the **compound** score.

**Before you run it:** in "This book is good.", only "good" is known (+1.9). Will the final score be higher or lower than 1.9?

In [ ]:
#@title The method: VADER's score for a sentence (from -1 to +1)
def score(sentence):
    # "compound" = VADER's overall score, brought back between -1 and +1
    return analyzer.polarity_scores(sentence)["compound"]

print(analyzer.polarity_scores("This book is good."))   # VADER's full answer
print(score("This book is good."))

**What you should see:** first VADER's full answer, `{'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.4404}` (the proportions of negative, neutral and positive text, then the compound score), then `0.4404`. The calculation: 1.9 / √(1.9² + 15) = 1.9 / 4.31 ≈ 0.44. The score is lower than 1.9: the formula squeezes every total between −1 and +1.

**Question:** with this formula, what total is needed to reach a score of 0.9? Is one very positive word enough? (Try it in step 9 with several positive words.)

## Step 8 · Negation and intensifiers: four versions

The same sentence in four versions: plain, strengthened by "very", negated by "not", then both at once.

**Before you run it:** rank the four sentences from the most negative to the most positive. Will "not very good" be more or less negative than "not good"?

In [ ]:
#@title Negation and intensifiers: four versions of a sentence
versions = ["This book is good.", "This book is very good.",
            "This book is not good.", "This book is not very good."]
for sentence in versions:
    print(score(sentence), "|", sentence)

**What you should see:** `0.4404` (good), `0.4927` (very good), `-0.3412` (not good), `-0.3865` (not very good). The negation does reverse the score, and "very" strengthens it, in both directions.

**Question:** for VADER, "not very good" is more negative than "not good". Is that your reading? In English, "not very good" is often a polite understatement, milder than "not good". What can such a simple rule not capture?

## Step 9 · Your turn: your sentence

Type your own sentence in the field `sentence`, then run the cell. It shows the split, the known words and the score. The sentence given is the one of the demonstration on the course site.

**Before you run it:** "beautiful" is worth +2.9, "disappointing" −2.2, strengthened by "really". Will the score be positive or negative?

In [ ]:
#@title TO CHANGE: your sentence
sentence = "This novel is beautiful, but the ending is really disappointing." #@param {type:"string"}
print("Words:", split_words(sentence))
print("Known words:", known_words(sentence))
print("Score:", score(sentence))

**What you should see:** `Known words: [('novel', 1.3), ('beautiful', 2.9), ('disappointing', -2.2)]` and `Score: -0.3898`, negative. Without its rules, the sum would be positive (1.3 + 2.9 − 2.2 = 2.0). VADER reads it as negative because the words after "but" count 1.5 times more and those before it only half, and "really" strengthens "disappointing". Notice that "novel" counts as a positive word.

**Question:** do you agree with VADER here? Then try an ironic sentence, an understatement ("it's not bad"), and the same sentence in CAPITAL LETTERS or with "!!!": what happens to the score?

## Step 10 · The score of the 20 reviews

We apply VADER to the 20 reviews, then turn each score into a label with a **threshold**: above +threshold, positive; below −threshold, negative; in between, neutral. The authors of VADER recommend 0.05; the slider `threshold` lets you change it.

**Before you run it:** how many reviews will the tool label as neutral? Think of the reviews in which VADER knows no word.

In [ ]:
#@title TO CHANGE: the neutrality threshold, then the score of each review
threshold = 0.05 #@param {type:"slider", min:0, max:0.5, step:0.05}
def label(s):
    # above the threshold: positive; below minus the threshold: negative; else: neutral
    if s >= threshold:
        return "positive"
    if s <= -threshold:
        return "negative"
    return "neutral"

df["score"] = df["text"].apply(score)
df["tool"] = df["score"].apply(label)
df

**What you should see:** the table of the 20 reviews with two more columns, `score` and `tool`. The most positive is review 19 (0.7868, "A sad story, but so beautiful."), the most negative review 18 (−0.8316, "The book tells of the father's death in the war."). 4 reviews score 0.0 and are labelled neutral: review 5 and the three missing-word reviews 13 to 15. Review 4 ("The novel has 300 pages…") scores 0.3182: positive, because of "novel". The ironic review 10 ("Great, another 900 pages…") gets 0.6249: positive.

**Question:** find in the table a review where the tool is right for a wrong reason, or by chance. (Hint: review 5 contains no known word; and in review 1, is "gripping" in the lexicon? Check it in step 5.)

## Step 11 · Chart 1: the distribution of sentiments

The chart asked for by the mini-project: how many positive, neutral and negative reviews, according to your reading (purple) and according to the tool (orange). The table of figures appears above it.

**Before you run it:** will the tool find more, fewer, or as many positive reviews as you?

In [ ]:
#@title Chart 1: the distribution of sentiments, according to you and to the tool
order = ["negative", "neutral", "positive"]
counts = pd.DataFrame({"your reading": df["manual"].value_counts(),
                       "the tool": df["tool"].value_counts()})
counts = counts.reindex(order).fillna(0).astype(int)   # same order; 0 if missing
print(counts)
counts.plot(kind="bar", rot=0, color=["#4a3aa7", "#eb6834"], figsize=(6, 3.5))
plt.title("Distribution of sentiments (20 fictional reviews)")
plt.ylabel("number of reviews")
plt.xlabel("")
plt.show()

**What you should see:** you: 9 negative, 3 neutral, 8 positive; the tool: 6 negative, 4 neutral, 10 positive. The tool sees the corpus as more positive than you do.

**Question:** if a report showed only this chart, could it tell which reviews the tool got wrong? Why do we need to compare review by review?

## Step 12 · Chart 2: the score of each review

One bar per review: to the right if the score is positive, to the left if it is negative. The color gives **your** label: blue = positive, gray = neutral, red = negative. The dotted lines mark the threshold.

**Before you run it:** what does a blue bar pointing to the left mean?

In [ ]:
#@title Chart 2: the score of each review, colored by your reading
colors = {"positive": "#2a78d6", "neutral": "#8f8d87", "negative": "#e34948"}
fig, ax = plt.subplots(figsize=(8, 7))
for tag, color in colors.items():
    part = df[df["manual"] == tag]
    ax.barh(part.index, part["score"], color=color, label="read as " + tag)
ax.set_yticks(df.index, df.index.astype(str) + ". " + df["text"].str[:34])
ax.invert_yaxis()                                   # review 0 at the top
ax.axvline(0, color="black", linewidth=0.8)
ax.vlines([-threshold, threshold], -0.5, len(df) - 0.5, colors="gray", linestyles="dotted")
ax.set_xlim(-1, 1)
ax.set_xlabel("score given by the tool (from -1 to +1)")
ax.set_title("Score of each review (fictional data)")
ax.legend(loc="center left")
plt.show()

**What you should see:** 20 bars. At the top, the plain reviews mostly point the right way, but the gray review 4 ("The novel has 300 pages…") points to the right. Lower down, the three ironic reviews (10 to 12) are red but point to the right; reviews 17 and 18 (war, death) point far to the left although 17 is blue and 18 gray; reviews 13 to 15 have no bar at all (score 0), while review 16 (the emoticon) points to the right.

**Question:** a blue bar pointing left, or a red bar pointing right, is an error of the tool. How many do you count? And the reviews without a bar: are they errors?

## Step 13 · Accuracy and confusion table

**Accuracy** is the share of reviews where the tool gives the same label as you. The **confusion table** crosses the two readings: one row per manual label, one column per label given by the tool. The right answers are on the diagonal.

**Before you run it:** judging from the charts, will the accuracy be closer to 30 %, 50 % or 80 %?

In [ ]:
#@title Accuracy: does the tool agree with your reading?
df["agree"] = df["manual"] == df["tool"]   # True if the tool gives the same label as you
print("Reviews correctly labelled:", df["agree"].sum(), "out of", len(df))
print("Accuracy:", round(100 * df["agree"].mean()), "%")
# Confusion table: rows = your label, columns = the tool's label
pd.crosstab(df["manual"], df["tool"])

**What you should see:** `Reviews correctly labelled: 10 out of 20` and `Accuracy: 50 %`. On the diagonal of the table: 4 negative, 1 neutral, 5 positive reviews correctly labelled. Out of the 9 negative reviews, 4 are labelled positive; out of the 3 neutral ones, only 1 is labelled neutral.

Careful: this corpus was built **to trap** the tool (half of the reviews are traps). On ordinary reviews, the accuracy would be higher; on your real corpus, you will have to measure it.

**Question:** a tool that always answered "negative" would get 9 right answers out of 20 (45 %). Does VADER do much better? What should always be given next to an accuracy figure?

## Step 14 · Accuracy by difficulty

We group the reviews by expected difficulty (column `difficulty`) and compute the accuracy of each group, as a percentage.

**Before you run it:** which group will have the best accuracy? Which one will get 0 %?

In [ ]:
#@title Accuracy by expected difficulty (in %)
by_difficulty = df.groupby("difficulty", sort=False)["agree"].mean() * 100
by_difficulty.round()

**What you should see:** `plain 83.0`, `negation 75.0`, `irony 0.0`, `missing word 25.0`, `sad theme 33.0` (in %). On the 10 ordinary reviews, VADER gets 8 right; on the 10 tricky reviews, only 2.

**Question:** the overall accuracy (50 %) hid these differences. If your real corpus contains a lot of irony (comments on a failed film adaptation, for example), what can you expect?

## Step 15 · Error analysis

Figures are not enough: we must reread the errors. For each wrongly labelled review, the cell shows its difficulty, your label, the tool's label, and the words that VADER found in its lexicon.

**Before you run it:** for the review "I cannot say that this book is good.", why did the negation not work? (Count the words between "cannot" and "good".)

In [ ]:
#@title Error analysis: reread each wrongly labelled review
errors = df[df["agree"] == False]
print(len(errors), "wrongly labelled reviews\n")
for i, row in errors.iterrows():
    print(i, "[" + str(row["difficulty"]) + "]", row["text"])
    print("   you:", row["manual"], "| tool:", row["tool"], "| score:", row["score"])
    print("   known words:", known_words(row["text"]))

**What you should see:** 10 wrongly labelled reviews. They fall into five families:
- **a word with two meanings** (review 4): "novel" is scored +1.3 as the adjective (new), so a neutral description becomes positive (0.3182);
- **distant negation** (review 9): "cannot" is 5 words before "good"; VADER only looks at the 3 words before, so "good" stays positive (0.4404);
- **irony** (reviews 10, 11, 12): "great", "well", "joy" are positive, but the reader means the opposite. The meaning comes from the rest of the sentence (900 pages where nothing happens, falling asleep on page 10) and from what we know about readers: no lexicon has access to it;
- **words missing from the lexicon** (reviews 13, 14, 15): no known word, score 0, so "neutral". Here the tool's "neutral" is ignorance, not neutrality. Review 16 is correct only because VADER knows the emoticon `:)`;
- **sad theme** (reviews 17 and 18): "war", "fear", "death" describe the story, not the reader's judgment. "a masterpiece" (+3.1) does not weigh enough against three negative words.

**Question:** for each family, suggest an improvement of the method (a rule, words to add, a wider window…). Which one seems impossible to fix with a lexicon?

## Step 16 · Add a word to the lexicon

The easiest thing to fix: missing words. Type a word in `new_word`, choose its score with the slider, then run the cell: the word enters VADER's lexicon and the 20 reviews are scored again. The change stays in VADER's memory: to go back to the original lexicon, run the notebook again from step 1 (in Colab: "Runtime → Restart session and run all").

**Before you run it:** with "page-turner" at +2.5, how many reviews will be correctly labelled?

In [ ]:
#@title TO CHANGE: add a word to the lexicon, then score everything again
new_word = "page-turner" #@param {type:"string"}
new_score = 2.5 #@param {type:"slider", min:-4, max:4, step:0.5}
LEXICON[new_word.lower()] = new_score
df["score"] = df["text"].apply(score)
df["tool"] = df["score"].apply(label)
df["agree"] = df["manual"] == df["tool"]
print("Words in the lexicon:", len(LEXICON))
print("Reviews correctly labelled:", df["agree"].sum(), "out of", len(df))
print("Accuracy:", round(100 * df["agree"].mean()), "%")

**What you should see:** `Words in the lexicon: 7507`, `Reviews correctly labelled: 11 out of 20`, `Accuracy: 55 %`: review 13 ("A page-turner: I devoured it in one night!") is now positive. If you then add "breathless" (+2) and "predictable" (−1.5), you reach 12, then 13 correct reviews (65 %).

**Question:** by adding the words missing from **these** 20 reviews, we improve the score… on these 20 reviews. Is it an honest improvement? What will happen on new reviews? (This is the problem of overfitting, chapter 3.)

## Step 17 · And VADER on French?

The course says it: VADER's lexicon is English. The cell makes it read three reviews in English, then their French translation. This matters if your corpus mixes languages, for instance to compare the reception of a novel in the original and in translation (comparative literature).

**Before you run it:** go back to your prediction in step 1. What scores will VADER give the French sentences?

In [ ]:
#@title VADER reads three reviews in English, then their French translation
pairs = [("A magnificent book, I adored it.", "Un roman magnifique, je l'ai adoré."),
         ("Boring book, and far too long.", "Livre ennuyeux et beaucoup trop long."),
         ("I did not like this book.", "Je n'ai pas aimé ce livre.")]
for english, french in pairs:
    print("VADER:", score(english), "|", english)
    print("VADER:", score(french), "|", french)
    print()

**What you should see:** in English, VADER gives 0.7717, −0.3182 and −0.2755; in French, 0.0 for all three sentences: it recognizes no word at all.

**Question:** the course suggests three solutions for French or Arabic: a multilingual model, TextBlob with its French extension, or machine translation into English before VADER. What risk does translation create for the tone of a review?

## Step 18 (optional) · A multilingual Transformer

> **Online cell.** It needs an internet connection; it works in Colab. It downloads a model of several hundred MB (one or two minutes the first time). No other cell depends on it: you can skip it.

The model `nlptown/bert-base-multilingual-uncased-sentiment` is a BERT-type Transformer (chapter 3), fine-tuned on product reviews in six languages, including English and French (not Arabic). It uses no lexicon: it has **learned** from annotated examples, and gives each review a rating from 1 to 5 stars. The cell turns the stars into labels (1-2 = negative, 3 = neutral, 4-5 = positive) and computes its accuracy.

**Before you run it:** will the Transformer do better than VADER's 50 %? On which traps?

In [ ]:
#@title (Online) A multilingual Transformer rates each review from 1 to 5 stars
from transformers import pipeline
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"   # model from the Hugging Face Hub
classifier = pipeline("sentiment-analysis", model=model_name)
stars = []
for text in df["text"]:
    result = classifier(text)[0]            # for example {'label': '4 stars', 'score': 0.52}
    stars.append(int(result["label"][0]))   # "4 stars" -> 4
bert_reviews = pd.DataFrame({"text": df["text"], "manual": df["manual"], "stars": stars})
to_label = {1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive"}
bert_reviews["model"] = bert_reviews["stars"].map(to_label)   # 1-2 stars = negative...
accuracy = (bert_reviews["model"] == bert_reviews["manual"]).mean()
print("Accuracy of the Transformer:", round(100 * accuracy), "%")
bert_reviews

**What you should see:** after the download, `Accuracy of the Transformer: … %`, then the table of the 20 reviews with their number of stars. The exact figures may change from one version to another: write down yours. Look especially at the ironic reviews (10 to 12) and at the missing-word reviews (13 to 16).

**Question:** the Transformer may be more accurate on some traps, but it cannot **explain** its ratings, whereas VADER shows each word and each rule. For a dissertation, which do you prefer: a more accurate tool, or a transparent one? Why?

## Your turn

Change only the fields, then run the cells again from the one you changed (in Colab: "Runtime → Run after").
1. **Step 10:** set the threshold to 0.3, then run steps 10 to 15 again. Reviews 6 and 11 become neutral: why? The accuracy goes down: in which case could a more cautious threshold still be useful?
2. **Step 16:** add, one by one, "breathless" (+2), "predictable" (−1.5), "cardboard" (−1). How high does the accuracy go? Then add "novel" with the score 0: what happens to review 4?
3. **Step 9:** write an ironic review that fools VADER, then a clearly positive review with no word from the lexicon. Keep them as examples for your report.

## For the mini-project: your own data

For your mini-project, replace the fictional reviews with **your** corpus:
- prepare a CSV file (in a spreadsheet: "Save as… → CSV UTF-8") with a column `text` (one text per row) and a column `manual`: your annotation, written exactly `positive`, `negative` or `neutral`. A column `difficulty` is optional. The separator can be a comma or a semicolon;
- public texts, collected in accordance with the site's terms of use, without usernames (in Algeria, law no. 18-07 of 10 June 2018 regulates personal data); keep the source and the date of each text separately;
- tick the box below and run the cell: in Colab, a window asks you for the file; in Jupyter, put the file next to the notebook under the name `my_corpus.csv`;
- then run steps 10 to 16 again: scores, charts, accuracy, errors.

**Before you run it:** if you leave the box unticked, what will the cell do?

In [ ]:
#@title TO CHANGE: load your own corpus (CSV file)
load_my_file = False #@param {type:"boolean"}
if load_my_file:
    try:
        from google.colab import files        # only exists in Colab
        upload = files.upload()               # window to choose the file
        file_name = list(upload.keys())[0]
    except ImportError:                       # Jupyter: file placed next to the notebook
        file_name = "my_corpus.csv"
    df = pd.read_csv(file_name, sep=None, engine="python")   # guesses , or ;
    if "difficulty" not in df.columns:
        df["difficulty"] = "not given"
    print(len(df), "texts loaded; columns:", list(df.columns))
else:
    print("Box not ticked: the notebook keeps the fictional reviews.")

**What you should see:** with the box unticked, `Box not ticked: the notebook keeps the fictional reviews.` With your file, the number of texts loaded and the list of your columns, which must contain `text` and `manual`.

**Question:** the course asks you to annotate a sample of about fifty texts by hand. Why not annotate the whole corpus? And why not skip annotation altogether?

## To hand in

Answer in writing, in English, in a few lines for each question.
1. **Measuring.** Give VADER's accuracy on the corpus and by difficulty (steps 13 and 14). Why is the overall accuracy not enough to judge the tool?
2. **Explaining the errors.** Choose three wrongly labelled reviews from different families (two meanings, negation, irony, missing word, sad theme). For each one, explain the error using the known words and VADER's rules.
3. **In your field.** "A score measures the color of the words used, not the quality of a critical judgment" (course, mini-project 1). Discuss this sentence using one review of the corpus, then explain how you would check the tool on your own corpus: reception of a novel or of its film adaptation (ANG, LGC), learners' comments or feedback on a course (DID).

## Key points

- A **lexicon-based method** such as VADER adds up the scores of rated words, with a few rules (negation, intensifiers, "but", threshold): each score can be explained; it is transparent symbolic AI.
- A tool is always evaluated against a **manual annotation**: accuracy, confusion table, then rereading the errors. The same overall distribution can hide many errors.
- Its limits are well known: **irony**, **distant negation**, **missing words**, words with **two meanings**, the **theme** mistaken for the judgment, and languages it was not built for (VADER does not read French).

**Lessons on the course site:** [Mini-project, subject 1: sentiment analysis](https://progremer04.github.io/canva-au-for-ang-/en/#mp-sujet-1) · [Chapter 3: symbolic AI](https://progremer04.github.io/canva-au-for-ang-/en/#c3-symbolique) · [Chapter 3: deep learning and Transformers](https://progremer04.github.io/canva-au-for-ang-/en/#c3-deep)